In [1]:
"""
benchmark_thctnet.py
────────────────────
Measures latency, throughput, and memory usage of THCT-Net
at frame lengths [5, 50, 100, 200, 300, 400, 500, 600, 750]
on both CPU and GPU (GPU skipped gracefully if unavailable).

Usage
─────
    python benchmark_thctnet.py

Outputs
───────
  • Console table (rich)
  • results/benchmark_original.csv
  • results/benchmark_original.png   (latency + memory charts)

Requirements
────────────
    pip install torch rich matplotlib pandas psutil
"""

import gc
import math
import os
import time
from typing import Dict, List

import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

# ── optional rich ────────────────────────────────────────────────────────────
try:
    from rich.console import Console
    from rich.table  import Table
    RICH = True
except ImportError:
    RICH = False

# ── optional matplotlib ──────────────────────────────────────────────────────
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    MPL = True
except ImportError:
    MPL = False

os.makedirs("results", exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Paste the model here so the script is self-contained
# (identical to the provided model.py minus the __main__ block)
# ─────────────────────────────────────────────────────────────────────────────
T_FRAMES     = 750
NUM_FEATURES = 132
M_ENTITIES   = 2
V_PER_ENT    = 22
C_IN         = 3
NUM_JOINTS   = V_PER_ENT * M_ENTITIES  # 44


class CausalLN2d(nn.Module):
    def __init__(self, num_channels: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.ln = nn.LayerNorm(num_channels, eps=eps)

    def forward(self, x: Tensor) -> Tensor:
        x = x.permute(0, 2, 3, 1)
        x = self.ln(x)
        return x.permute(0, 3, 1, 2)


class CausalLN3d(nn.Module):
    def __init__(self, num_channels: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.ln = nn.LayerNorm(num_channels, eps=eps)

    def forward(self, x: Tensor) -> Tensor:
        x = x.permute(0, 2, 3, 4, 1)
        x = self.ln(x)
        return x.permute(0, 4, 1, 2, 3)


class CausalLN1d(nn.Module):
    def __init__(self, num_channels: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.ln = nn.LayerNorm(num_channels, eps=eps)

    def forward(self, x: Tensor) -> Tensor:
        x = x.permute(0, 2, 1)
        x = self.ln(x)
        return x.permute(0, 2, 1)


def _causal_pad1d(x, kernel_size, dilation=1):
    return F.pad(x, ((kernel_size - 1) * dilation, 0))


def _causal_pad2d_time(x, kernel_t, dilation_t=1):
    return F.pad(x, (0, 0, (kernel_t - 1) * dilation_t, 0))


def _to_skeleton_tensor(x: Tensor) -> Tensor:
    B, T, D = x.shape
    x = x.reshape(B, T, M_ENTITIES, V_PER_ENT, C_IN)
    return x.permute(0, 4, 1, 3, 2).contiguous()


class _CausalISATABlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Identity()
        self.alpha  = nn.Parameter(torch.ones(1))
        self.A      = nn.Parameter(torch.zeros(num_heads, 1, 1))
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        B, U, d = x.shape
        H, Dh   = self.num_heads, self.head_dim
        residual = x
        x_n = self.norm1(x)
        Q = self.q_proj(x_n).reshape(B, U, H, Dh).transpose(1, 2)
        K = self.k_proj(x_n).reshape(B, U, H, Dh).transpose(1, 2)
        V = x_n.reshape(B, U, H, Dh).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Dh)
        scores = self.alpha * torch.tanh(scores) + self.A
        mask   = torch.triu(torch.full((U, U), float("-inf"), device=x.device), diagonal=1)
        scores = scores + mask
        attn = F.softmax(scores, dim=-1)
        out  = torch.matmul(attn, V).transpose(1, 2).reshape(B, U, d) + residual
        return out + self.ffn(self.norm2(out))


class TransformerStream(nn.Module):
    def __init__(self, num_classes, d_model=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.frame_embed = nn.Sequential(
            nn.Conv3d(C_IN, d_model, kernel_size=(1, V_PER_ENT, M_ENTITIES),
                      stride=(1, V_PER_ENT, M_ENTITIES), bias=False),
            CausalLN3d(d_model), nn.GELU(),
        )
        self.pos_enc    = nn.Parameter(torch.zeros(1, T_FRAMES, d_model))
        nn.init.trunc_normal_(self.pos_enc, std=0.02)
        self.blocks     = nn.ModuleList([_CausalISATABlock(d_model, num_heads, dropout)
                                         for _ in range(num_layers)])
        self.temporal_agg = nn.Conv1d(d_model, d_model, kernel_size=5, bias=False)
        self.agg_norm     = CausalLN1d(d_model)
        self.head         = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B, C, T, V, M = x.shape
        tokens = self.frame_embed(x).squeeze(-1).squeeze(-1).transpose(1, 2)
        tokens = tokens + self.pos_enc[:, :T, :]
        for blk in self.blocks:
            tokens = blk(tokens)
        t = tokens.transpose(1, 2)
        t = _causal_pad1d(t, kernel_size=5)
        t = F.gelu(self.agg_norm(self.temporal_agg(t)))
        return self.head(t.transpose(1, 2))


class _CausalCNNBranch(nn.Module):
    def __init__(self, base_ch=64):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(C_IN, base_ch, kernel_size=(1, 1), bias=False),
            CausalLN2d(base_ch), nn.ReLU(inplace=True),
        )
        self.enc2_conv = nn.Conv2d(base_ch, base_ch, kernel_size=(3, 1), padding=0, bias=False)
        self.enc2_norm = CausalLN2d(base_ch)
        self.enc3_conv = nn.Conv2d(NUM_JOINTS, base_ch, kernel_size=(3, 3),
                                   padding=(0, 1), dilation=(2, 1), bias=False)
        self.enc3_norm = CausalLN2d(base_ch)
        self.enc4_conv = nn.Conv2d(base_ch, base_ch, kernel_size=(3, 3),
                                   padding=(0, 1), dilation=(1, 1), bias=False)
        self.enc4_norm = CausalLN2d(base_ch)

    def forward(self, x):
        x = self.enc1(x)
        x = _causal_pad2d_time(x, kernel_t=3, dilation_t=1)
        x = F.relu(self.enc2_norm(self.enc2_conv(x)))
        x = x.permute(0, 3, 2, 1).contiguous()
        x = _causal_pad2d_time(x, kernel_t=3, dilation_t=2)
        x = F.relu(self.enc3_norm(self.enc3_conv(x)))
        x = _causal_pad2d_time(x, kernel_t=3, dilation_t=1)
        return F.relu(self.enc4_norm(self.enc4_conv(x)))


class _CausalResidualFusion(nn.Module):
    def __init__(self, in_ch, out_ch=128):
        super().__init__()
        self.conv_s = nn.Conv2d(in_ch, out_ch, kernel_size=(1, 7), padding=(0, 3), bias=False)
        self.norm_s = CausalLN2d(out_ch)
        self.conv_t = nn.Conv2d(out_ch, out_ch, kernel_size=(7, 1), padding=0, bias=False)
        self.norm_t = CausalLN2d(out_ch)
        self.shortcut = (nn.Sequential(nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                                       CausalLN2d(out_ch))
                         if in_ch != out_ch else nn.Identity())
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        skip = self.shortcut(x)
        out  = F.relu(self.norm_s(self.conv_s(x)))
        out  = _causal_pad2d_time(out, kernel_t=7)
        return self.act(self.norm_t(self.conv_t(out)) + skip)


class CNNStream(nn.Module):
    def __init__(self, num_classes, base_ch=64):
        super().__init__()
        self.branch_S = _CausalCNNBranch(base_ch)
        self.branch_M = _CausalCNNBranch(base_ch)
        self.fusion   = _CausalResidualFusion(base_ch * 2, out_ch=128)
        self.fc1  = nn.Linear(128, 256)
        self.drop = nn.Dropout(0.3)
        self.fc2  = nn.Linear(256, num_classes)

    @staticmethod
    def _backward_diff(x):
        diff = x[:, :, 1:, :] - x[:, :, :-1, :]
        return F.pad(diff, (0, 0, 1, 0))

    def forward(self, x):
        B, C, T, V, M = x.shape
        x_flat   = x.reshape(B, C, T, V * M)
        x_motion = self._backward_diff(x_flat)
        feat_S   = self.branch_S(x_flat)
        feat_M   = self.branch_M(x_motion)
        if feat_S.shape != feat_M.shape:
            feat_M = F.interpolate(feat_M, size=feat_S.shape[2:])
        fused = torch.cat([feat_S, feat_M], dim=1)
        fused = self.fusion(fused)
        out   = fused.mean(dim=-1).transpose(1, 2)
        return self.fc2(self.drop(F.relu(self.fc1(out))))


class THCTNet(nn.Module):
    def __init__(self, num_classes, d_model=128, num_heads=4, num_layers=4, base_ch=64):
        super().__init__()
        self.num_classes  = num_classes
        self.cnn_stream   = CNNStream(num_classes, base_ch)
        self.trans_stream = TransformerStream(num_classes, d_model, num_heads, num_layers)
        self._raw_w       = nn.Parameter(torch.zeros(1))

    def forward(self, sequences, lengths=None):
        x = _to_skeleton_tensor(sequences)
        lc = self.cnn_stream(x)
        lt = self.trans_stream(x)
        w  = torch.sigmoid(self._raw_w)
        return w * lc + (1.0 - w) * lt


# ─────────────────────────────────────────────────────────────────────────────
# Benchmark helpers
# ─────────────────────────────────────────────────────────────────────────────
FRAME_LENGTHS = [5, 50, 100, 200, 300, 400, 500, 600, 750]
NUM_CLASSES   = 21
BATCH_SIZE    = 1          # typical inference batch
WARMUP_RUNS   = 3
TIMED_RUNS    = 10


def _bytes_to_mb(b: int) -> float:
    return b / (1024 ** 2)


def get_process_ram_mb() -> float:
    """RSS of current process in MB."""
    return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)


def count_params(model: nn.Module):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def benchmark_one(model: nn.Module, T: int, device: torch.device) -> Dict:
    model.eval()
    seq = torch.randn(BATCH_SIZE, T, NUM_FEATURES, device=device)

    # ── Warmup ───────────────────────────────────────────────────────
    for _ in range(WARMUP_RUNS):
        with torch.no_grad():
            _ = model(seq)
    if device.type == "cuda":
        torch.cuda.synchronize(device)

    # ── Peak GPU memory reset ────────────────────────────────────────
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    # ── RAM before ───────────────────────────────────────────────────
    ram_before = get_process_ram_mb()

    # ── Timed runs ───────────────────────────────────────────────────
    latencies = []
    for _ in range(TIMED_RUNS):
        if device.type == "cuda":
            start_ev = torch.cuda.Event(enable_timing=True)
            end_ev   = torch.cuda.Event(enable_timing=True)
            start_ev.record()
            with torch.no_grad():
                out = model(seq)
            end_ev.record()
            torch.cuda.synchronize(device)
            latencies.append(start_ev.elapsed_time(end_ev))   # ms
        else:
            t0 = time.perf_counter()
            with torch.no_grad():
                out = model(seq)
            latencies.append((time.perf_counter() - t0) * 1000)  # ms

    latency_mean = sum(latencies) / len(latencies)
    latency_std  = (sum((l - latency_mean) ** 2 for l in latencies) / len(latencies)) ** 0.5
    latency_min  = min(latencies)
    latency_max  = max(latencies)

    # ── RAM after ────────────────────────────────────────────────────
    ram_after      = get_process_ram_mb()
    ram_delta_mb   = max(ram_after - ram_before, 0.0)

    # ── GPU memory ───────────────────────────────────────────────────
    peak_gpu_mb = 0.0
    if device.type == "cuda":
        peak_gpu_mb = _bytes_to_mb(torch.cuda.max_memory_allocated(device))

    # ── Throughput ───────────────────────────────────────────────────
    throughput_fps = (BATCH_SIZE * T * 1000) / latency_mean   # frames/sec

    return {
        "T":               T,
        "latency_mean_ms": round(latency_mean, 3),
        "latency_std_ms":  round(latency_std,  3),
        "latency_min_ms":  round(latency_min,  3),
        "latency_max_ms":  round(latency_max,  3),
        "throughput_fps":  round(throughput_fps, 1),
        "peak_gpu_mb":     round(peak_gpu_mb, 2),
        "ram_delta_mb":    round(ram_delta_mb, 2),
        "output_shape":    tuple(out.shape),
    }


def run_benchmark(label: str = "original") -> pd.DataFrame:
    total_p, train_p = count_params(THCTNet(NUM_CLASSES))
    print(f"\n{'='*64}")
    print(f"  THCT-Net  ({label})   — {total_p:,} params ({train_p:,} trainable)")
    print(f"{'='*64}")

    devices_to_test = [torch.device("cpu")]
    if torch.cuda.is_available():
        devices_to_test.append(torch.device("cuda"))

    rows: List[Dict] = []

    for device in devices_to_test:
        dev_name = (torch.cuda.get_device_name(device)
                    if device.type == "cuda" else "CPU")
        print(f"\n  Device : {dev_name} ({device})")
        print(f"  {'T':>6}  {'mean ms':>10}  {'std ms':>8}  {'min ms':>8}  "
              f"{'max ms':>8}  {'FPS':>10}  {'peak GPU MB':>12}  {'RAM ΔMIB':>10}")
        print(f"  {'-'*6}  {'-'*10}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*10}  {'-'*12}  {'-'*10}")

        # build fresh model per device so weights land on the right device
        model = THCTNet(NUM_CLASSES).to(device)
        model.eval()

        for T in FRAME_LENGTHS:
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
            try:
                res = benchmark_one(model, T, device)
            except RuntimeError as e:
                print(f"  T={T:4d}  OOM / error: {e}")
                continue

            print(f"  {res['T']:>6}  {res['latency_mean_ms']:>10.3f}  "
                  f"{res['latency_std_ms']:>8.3f}  {res['latency_min_ms']:>8.3f}  "
                  f"{res['latency_max_ms']:>8.3f}  {res['throughput_fps']:>10.1f}  "
                  f"{res['peak_gpu_mb']:>12.2f}  {res['ram_delta_mb']:>10.2f}")

            rows.append({
                "model":           label,
                "device":          device.type,
                "device_name":     dev_name,
                **res,
            })

        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    csv_path = f"results/benchmark_{label}.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n  Results saved → {csv_path}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Plotting
# ─────────────────────────────────────────────────────────────────────────────
def plot_results(df: pd.DataFrame, label: str = "original"):
    if not MPL:
        return

    devices = df["device"].unique()
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"THCT-Net Benchmark  ({label})", fontsize=14, fontweight="bold")

    colors = {"cpu": "#4C72B0", "cuda": "#DD8452"}

    # (0,0) Latency mean
    ax = axes[0, 0]
    for dev in devices:
        sub = df[df["device"] == dev]
        c   = colors.get(dev, "gray")
        ax.plot(sub["T"], sub["latency_mean_ms"], marker="o", label=dev.upper(), color=c)
        ax.fill_between(sub["T"],
                        sub["latency_mean_ms"] - sub["latency_std_ms"],
                        sub["latency_mean_ms"] + sub["latency_std_ms"],
                        alpha=0.15, color=c)
    ax.set_title("Mean Inference Latency")
    ax.set_xlabel("Frame Length (T)")
    ax.set_ylabel("Latency (ms)")
    ax.legend(); ax.grid(True, alpha=0.3)

    # (0,1) Throughput
    ax = axes[0, 1]
    for dev in devices:
        sub = df[df["device"] == dev]
        ax.plot(sub["T"], sub["throughput_fps"], marker="s",
                label=dev.upper(), color=colors.get(dev, "gray"))
    ax.set_title("Throughput")
    ax.set_xlabel("Frame Length (T)")
    ax.set_ylabel("Frames / second")
    ax.legend(); ax.grid(True, alpha=0.3)

    # (1,0) Peak GPU memory
    ax = axes[1, 0]
    cuda_df = df[df["device"] == "cuda"]
    if not cuda_df.empty:
        ax.plot(cuda_df["T"], cuda_df["peak_gpu_mb"], marker="^",
                color=colors["cuda"], label="CUDA Peak")
        ax.set_ylabel("Peak GPU Memory (MB)")
    else:
        ax.text(0.5, 0.5, "No CUDA device", ha="center", va="center",
                transform=ax.transAxes, fontsize=12, color="gray")
    ax.set_title("Peak GPU Memory Allocated")
    ax.set_xlabel("Frame Length (T)")
    ax.legend(); ax.grid(True, alpha=0.3)

    # (1,1) Latency std (stability)
    ax = axes[1, 1]
    for dev in devices:
        sub = df[df["device"] == dev]
        ax.plot(sub["T"], sub["latency_std_ms"], marker="D",
                label=dev.upper(), color=colors.get(dev, "gray"))
    ax.set_title("Latency Std Dev (Stability)")
    ax.set_xlabel("Frame Length (T)")
    ax.set_ylabel("Std Dev (ms)")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = f"results/benchmark_{label}.png"
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"  Plot saved     → {path}")


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = run_benchmark(label="original")
    plot_results(df, label="original")
    print("\nDone. Run benchmark_kvcache.py next to compare with KV-cache.\n")


  THCT-Net  (original)   — 1,277,759 params (1,277,759 trainable)

  Device : CPU (cpu)
       T     mean ms    std ms    min ms    max ms         FPS   peak GPU MB    RAM ΔMIB
  ------  ----------  --------  --------  --------  ----------  ------------  ----------
       5      51.234    22.378    22.935   100.142        97.6          0.00        0.00
      50      96.049    50.688    54.995   196.440       520.6          0.00        0.00
     100      82.800     9.219    72.726   100.405      1207.7          0.00        0.01
     200     162.102    24.972   134.809   223.829      1233.8          0.00        0.05
     300     253.161    48.798   200.820   361.781      1185.0          0.00        0.00
     400     290.247    16.030   270.444   333.603      1378.1          0.00        0.00
     500     355.740     8.477   342.985   373.956      1405.5          0.00        0.09
     600     465.256    50.638   420.423   603.004      1289.6          0.00        3.23
     750     565.016 

In [2]:
"""
benchmark_kvcache.py
────────────────────
Adds a KV-cache to the THCT-Net transformer stream, then benchmarks
BOTH the original and the cached model side-by-side.

What the KV-cache does
──────────────────────
In the causal ISATA attention blocks each new token only needs to attend
to *all previous* K and V tensors, which never change.  Instead of
recomputing K and V for the full sequence on every call, we:

  1. Store a (B, H, T_seen, Dh) K cache and V cache per block.
  2. On a new token step, project only the new token's K and V, append
     them to the cache, then compute attention against the full cache.
  3. The FFN and LN are applied per-token as usual.

This turns O(T²) attention work into O(T·T_past) with zero allocations
for past tokens → significant speedup for autoregressive / streaming use.

Two benchmark modes
───────────────────
  • "batch" mode  : full sequence fed at once (like the original).
      KV-cache offers no speedup here — it just pre-fills the cache.
      Useful as a correctness check.

  • "streaming" mode : tokens fed one at a time, cache reused.
      This is the real use-case and shows the speedup.

Outputs
───────
  • results/benchmark_kvcache.csv
  • results/benchmark_kvcache.png
  • results/benchmark_comparison.csv   (original merged for easy diff)
  • results/benchmark_comparison.png

Usage
─────
    python benchmark_kvcache.py
"""

import gc
import math
import os
import time
from typing import Dict, List, Optional, Tuple

import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    MPL = True
except ImportError:
    MPL = False

os.makedirs("results", exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────────────────────────────────────
T_FRAMES     = 750
NUM_FEATURES = 132
M_ENTITIES   = 2
V_PER_ENT    = 22
C_IN         = 3
NUM_JOINTS   = V_PER_ENT * M_ENTITIES  # 44

FRAME_LENGTHS = [5, 50, 100, 200, 300, 400, 500, 600, 750]
NUM_CLASSES   = 21
BATCH_SIZE    = 1
WARMUP_RUNS   = 3
TIMED_RUNS    = 10


# ─────────────────────────────────────────────────────────────────────────────
# Shared building blocks (identical to original)
# ─────────────────────────────────────────────────────────────────────────────
class CausalLN2d(nn.Module):
    def __init__(self, c, eps=1e-5):
        super().__init__(); self.ln = nn.LayerNorm(c, eps=eps)
    def forward(self, x):
        return self.ln(x.permute(0,2,3,1)).permute(0,3,1,2)

class CausalLN3d(nn.Module):
    def __init__(self, c, eps=1e-5):
        super().__init__(); self.ln = nn.LayerNorm(c, eps=eps)
    def forward(self, x):
        return self.ln(x.permute(0,2,3,4,1)).permute(0,4,1,2,3)

class CausalLN1d(nn.Module):
    def __init__(self, c, eps=1e-5):
        super().__init__(); self.ln = nn.LayerNorm(c, eps=eps)
    def forward(self, x):
        return self.ln(x.permute(0,2,1)).permute(0,2,1)

def _causal_pad1d(x, ks, d=1):   return F.pad(x, ((ks-1)*d, 0))
def _causal_pad2d_time(x, kt, dt=1): return F.pad(x, (0, 0, (kt-1)*dt, 0))

def _to_skeleton_tensor(x: Tensor) -> Tensor:
    B, T, D = x.shape
    return x.reshape(B, T, M_ENTITIES, V_PER_ENT, C_IN).permute(0,4,1,3,2).contiguous()


# ─────────────────────────────────────────────────────────────────────────────
# KV-Cache container
# ─────────────────────────────────────────────────────────────────────────────
class KVCache:
    """
    Holds growing K and V tensors for one attention block.
    Shape after t steps: (B, H, t, Dh) for both k and v.
    """
    def __init__(self):
        self.k: Optional[Tensor] = None
        self.v: Optional[Tensor] = None

    def reset(self):
        self.k = None
        self.v = None

    def update(self, new_k: Tensor, new_v: Tensor) -> Tuple[Tensor, Tensor]:
        """Append new_k / new_v (B,H,1,Dh) and return full cache."""
        if self.k is None:
            self.k, self.v = new_k, new_v
        else:
            self.k = torch.cat([self.k, new_k], dim=2)
            self.v = torch.cat([self.v, new_v], dim=2)
        return self.k, self.v


# ─────────────────────────────────────────────────────────────────────────────
# KV-Cache aware ISATA block
# ─────────────────────────────────────────────────────────────────────────────
class _KVCausalISATABlock(nn.Module):
    """
    Causal ISATA block with optional KV-cache for single-token streaming.

    forward(x, cache=None)
      • x.shape == (B, T, d)   → batch-process T tokens (pre-fill / full-pass)
      • x.shape == (B, 1, d)   → single new token with cache
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.alpha  = nn.Parameter(torch.ones(1))
        self.A      = nn.Parameter(torch.zeros(num_heads, 1, 1))
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: Tensor, cache: Optional[KVCache] = None) -> Tensor:
        B, Tq, d = x.shape
        H, Dh    = self.num_heads, self.head_dim
        residual = x
        x_n = self.norm1(x)

        Q  = self.q_proj(x_n).reshape(B, Tq, H, Dh).transpose(1, 2)
        Kn = self.k_proj(x_n).reshape(B, Tq, H, Dh).transpose(1, 2)
        Vn = x_n             .reshape(B, Tq, H, Dh).transpose(1, 2)

        if cache is not None:
            K, V = cache.update(Kn, Vn)   # full past K/V
        else:
            K, V = Kn, Vn

        Tk = K.shape[2]   # total key length (past + present)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Dh)
        scores = self.alpha * torch.tanh(scores) + self.A

        # Causal mask: each query attends only to positions ≤ itself
        if Tq > 1:   # batch / prefill — need full Tq×Tk causal mask
            q_idx  = torch.arange(Tk - Tq, Tk, device=x.device).unsqueeze(1)   # (Tq, 1)
            k_idx  = torch.arange(Tk,       device=x.device).unsqueeze(0)       # (1,  Tk)
            causal = k_idx > q_idx                                               # (Tq, Tk)
            # torch.where avoids 0 * (-inf) = NaN from float-multiplication mask
            scores = torch.where(causal, torch.full_like(scores, float("-inf")), scores)

        # Tq==1: single new token always attends to all past → no mask needed

        attn = F.softmax(scores, dim=-1)
        out  = torch.matmul(attn, V).transpose(1, 2).reshape(B, Tq, d) + residual
        return out + self.ffn(self.norm2(out))


# ─────────────────────────────────────────────────────────────────────────────
# KV-Cache transformer stream
# ─────────────────────────────────────────────────────────────────────────────
class TransformerStreamKV(nn.Module):
    """
    Drop-in replacement for TransformerStream with KV-cache support.

    Two calling modes
    -----------------
    Batch (full sequence):
        logits = stream.forward(x)
        → works exactly like the original; cache is ignored.

    Streaming (token by token):
        stream.reset_cache(B, device)
        for t in range(T):
            token_t = x[:, :, t:t+1, :, :]        # (B, C, 1, V, M)
            logit_t = stream.forward_step(token_t)  # (B, 1, num_classes)
    """

    def __init__(self, num_classes, d_model=128, num_heads=4,
                 num_layers=4, dropout=0.1):
        super().__init__()
        self.d_model    = d_model
        self.num_layers = num_layers

        self.frame_embed = nn.Sequential(
            nn.Conv3d(C_IN, d_model, kernel_size=(1, V_PER_ENT, M_ENTITIES),
                      stride=(1, V_PER_ENT, M_ENTITIES), bias=False),
            CausalLN3d(d_model), nn.GELU(),
        )
        self.pos_enc    = nn.Parameter(torch.zeros(1, T_FRAMES, d_model))
        nn.init.trunc_normal_(self.pos_enc, std=0.02)
        self.blocks     = nn.ModuleList([
            _KVCausalISATABlock(d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.temporal_agg = nn.Conv1d(d_model, d_model, kernel_size=5, bias=False)
        self.agg_norm     = CausalLN1d(d_model)
        self.head         = nn.Linear(d_model, num_classes)

        # Cache state (set via reset_cache)
        self._caches:      List[KVCache] = [KVCache() for _ in range(num_layers)]
        self._agg_buf:     Optional[Tensor] = None   # (B, d, 4) for causal Conv1d context
        self._step_count:  int = 0

    # ── Batch forward (same as original) ─────────────────────────────────
    def forward(self, x: Tensor) -> Tensor:
        """(B, C, T, V, M) → (B, T, num_classes)"""
        B, C, T, V, M = x.shape
        tokens = self.frame_embed(x).squeeze(-1).squeeze(-1).transpose(1, 2)
        tokens = tokens + self.pos_enc[:, :T, :]
        for blk in self.blocks:
            tokens = blk(tokens, cache=None)          # no cache in batch mode
        t = tokens.transpose(1, 2)
        t = _causal_pad1d(t, 5)
        t = F.gelu(self.agg_norm(self.temporal_agg(t)))
        return self.head(t.transpose(1, 2))

    # ── Streaming helpers ─────────────────────────────────────────────────
    def reset_cache(self, B: int, device: torch.device) -> None:
        """Call before streaming a new sequence."""
        for c in self._caches:
            c.reset()
        self._agg_buf    = torch.zeros(B, self.d_model, 4, device=device)
        self._step_count = 0

    def forward_step(self, x_step: Tensor) -> Tensor:
        """
        Process a single new frame (B, C, 1, V, M) → (B, 1, num_classes).
        KV-caches are updated in-place for all attention blocks.
        """
        B = x_step.shape[0]
        t_idx = self._step_count

        token = self.frame_embed(x_step).squeeze(-1).squeeze(-1).transpose(1, 2)
        # (B, 1, d)
        token = token + self.pos_enc[:, t_idx:t_idx+1, :]

        for blk, cache in zip(self.blocks, self._caches):
            token = blk(token, cache=cache)         # (B, 1, d)

        # Causal Conv1d: maintain a 4-frame left buffer
        t_ch = token.transpose(1, 2)               # (B, d, 1)
        ctx  = torch.cat([self._agg_buf, t_ch], dim=2)   # (B, d, 5)
        self._agg_buf = ctx[:, :, 1:]              # keep last 4 as new buffer
        agg  = self.temporal_agg(ctx)              # (B, d, 1)  kernel=5, no pad needed
        agg  = F.gelu(self.agg_norm(agg))          # (B, d, 1)

        self._step_count += 1
        return self.head(agg.transpose(1, 2))      # (B, 1, num_classes)


# ─────────────────────────────────────────────────────────────────────────────
# CNN stream (unchanged from original)
# ─────────────────────────────────────────────────────────────────────────────
class _CausalCNNBranch(nn.Module):
    def __init__(self, base_ch=64):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(C_IN, base_ch, (1,1), bias=False),
            CausalLN2d(base_ch), nn.ReLU(inplace=True))
        self.enc2_conv = nn.Conv2d(base_ch, base_ch, (3,1), padding=0, bias=False)
        self.enc2_norm = CausalLN2d(base_ch)
        self.enc3_conv = nn.Conv2d(NUM_JOINTS, base_ch, (3,3), padding=(0,1), dilation=(2,1), bias=False)
        self.enc3_norm = CausalLN2d(base_ch)
        self.enc4_conv = nn.Conv2d(base_ch, base_ch, (3,3), padding=(0,1), bias=False)
        self.enc4_norm = CausalLN2d(base_ch)

    def forward(self, x):
        x = self.enc1(x)
        x = F.relu(self.enc2_norm(self.enc2_conv(_causal_pad2d_time(x, 3, 1))))
        x = x.permute(0,3,2,1).contiguous()
        x = F.relu(self.enc3_norm(self.enc3_conv(_causal_pad2d_time(x, 3, 2))))
        return F.relu(self.enc4_norm(self.enc4_conv(_causal_pad2d_time(x, 3, 1))))

class _CausalResidualFusion(nn.Module):
    def __init__(self, in_ch, out_ch=128):
        super().__init__()
        self.conv_s = nn.Conv2d(in_ch, out_ch, (1,7), padding=(0,3), bias=False)
        self.norm_s = CausalLN2d(out_ch)
        self.conv_t = nn.Conv2d(out_ch, out_ch, (7,1), padding=0, bias=False)
        self.norm_t = CausalLN2d(out_ch)
        self.shortcut = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), CausalLN2d(out_ch))
                         if in_ch != out_ch else nn.Identity())
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        skip = self.shortcut(x)
        out  = F.relu(self.norm_s(self.conv_s(x)))
        return self.act(self.norm_t(self.conv_t(_causal_pad2d_time(out, 7))) + skip)

class CNNStream(nn.Module):
    def __init__(self, num_classes, base_ch=64):
        super().__init__()
        self.branch_S = _CausalCNNBranch(base_ch)
        self.branch_M = _CausalCNNBranch(base_ch)
        self.fusion   = _CausalResidualFusion(base_ch * 2, 128)
        self.fc1 = nn.Linear(128, 256); self.drop = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, num_classes)

    @staticmethod
    def _backward_diff(x):
        return F.pad(x[:,:,1:,:] - x[:,:,:-1,:], (0,0,1,0))

    def forward(self, x):
        B,C,T,V,M = x.shape
        flat = x.reshape(B,C,T,V*M)
        mot  = self._backward_diff(flat)
        fS   = self.branch_S(flat); fM = self.branch_M(mot)
        if fS.shape != fM.shape: fM = F.interpolate(fM, size=fS.shape[2:])
        fused = self.fusion(torch.cat([fS, fM], dim=1))
        out   = fused.mean(dim=-1).transpose(1,2)
        return self.fc2(self.drop(F.relu(self.fc1(out))))


# ─────────────────────────────────────────────────────────────────────────────
# KV-Cache enabled top-level model
# ─────────────────────────────────────────────────────────────────────────────
class THCTNetKV(nn.Module):
    """
    THCT-Net with KV-cached transformer stream.

    Batch mode (full sequence):
        logits = model(sequences, lengths)     # identical API to THCTNet

    Streaming mode (token by token, transformer only):
        model.reset_cache(B, device)
        for t in range(T):
            frame_t  = sequences[:, t:t+1, :]         # (B, 1, D)
            skel_t   = _to_skeleton_tensor(frame_t)   # (B, C, 1, V, M)
            logit_t  = model.trans_stream.forward_step(skel_t)
    """

    def __init__(self, num_classes, d_model=128, num_heads=4, num_layers=4, base_ch=64):
        super().__init__()
        self.num_classes  = num_classes
        self.cnn_stream   = CNNStream(num_classes, base_ch)
        self.trans_stream = TransformerStreamKV(num_classes, d_model, num_heads, num_layers)
        self._raw_w       = nn.Parameter(torch.zeros(1))

    def forward(self, sequences: Tensor, lengths=None) -> Tensor:
        x  = _to_skeleton_tensor(sequences)
        lc = self.cnn_stream(x)
        lt = self.trans_stream(x)
        w  = torch.sigmoid(self._raw_w)
        return w * lc + (1.0 - w) * lt

    def reset_cache(self, B: int, device: torch.device):
        self.trans_stream.reset_cache(B, device)


# ─────────────────────────────────────────────────────────────────────────────
# Benchmark helpers
# ─────────────────────────────────────────────────────────────────────────────
def _bytes_to_mb(b): return b / (1024 ** 2)
def get_ram_mb():    return psutil.Process(os.getpid()).memory_info().rss / (1024**2)
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)


def benchmark_batch(model: nn.Module, T: int, device: torch.device) -> Dict:
    """Standard full-sequence forward pass."""
    model.eval()
    seq = torch.randn(BATCH_SIZE, T, NUM_FEATURES, device=device)
    for _ in range(WARMUP_RUNS):
        with torch.no_grad(): _ = model(seq)
    if device.type == "cuda": torch.cuda.synchronize(device)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats(device)
    ram0 = get_ram_mb()

    lats = []
    for _ in range(TIMED_RUNS):
        if device.type == "cuda":
            e0, e1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            e0.record()
            with torch.no_grad(): out = model(seq)
            e1.record(); torch.cuda.synchronize(device)
            lats.append(e0.elapsed_time(e1))
        else:
            t0 = time.perf_counter()
            with torch.no_grad(): out = model(seq)
            lats.append((time.perf_counter()-t0)*1000)

    m = sum(lats)/len(lats)
    s = (sum((l-m)**2 for l in lats)/len(lats))**0.5
    return {
        "T": T, "mode": "batch",
        "latency_mean_ms": round(m, 3), "latency_std_ms": round(s, 3),
        "latency_min_ms":  round(min(lats), 3), "latency_max_ms": round(max(lats), 3),
        "throughput_fps":  round(BATCH_SIZE*T*1000/m, 1),
        "peak_gpu_mb":     round(_bytes_to_mb(torch.cuda.max_memory_allocated(device)) if device.type=="cuda" else 0, 2),
        "ram_delta_mb":    round(max(get_ram_mb()-ram0, 0), 2),
    }


def benchmark_streaming(model: THCTNetKV, T: int, device: torch.device) -> Dict:
    """
    Token-by-token streaming using KV-cache for the transformer.
    CNN stream processes the full sequence once (it has no streaming API).
    We measure the *transformer stream only* in streaming mode.
    """
    model.eval()
    seq  = torch.randn(BATCH_SIZE, T, NUM_FEATURES, device=device)
    skel = _to_skeleton_tensor(seq)    # (B, C, T, V, M)

    # Warmup
    for _ in range(WARMUP_RUNS):
        model.trans_stream.reset_cache(BATCH_SIZE, device)
        with torch.no_grad():
            for t in range(T):
                _ = model.trans_stream.forward_step(skel[:, :, t:t+1, :, :])
    if device.type == "cuda": torch.cuda.synchronize(device)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats(device)
    ram0 = get_ram_mb()

    lats = []
    for _ in range(TIMED_RUNS):
        model.trans_stream.reset_cache(BATCH_SIZE, device)
        if device.type == "cuda":
            e0 = torch.cuda.Event(enable_timing=True)
            e1 = torch.cuda.Event(enable_timing=True)
            e0.record()
        else:
            t0 = time.perf_counter()
        with torch.no_grad():
            for t in range(T):
                _ = model.trans_stream.forward_step(skel[:, :, t:t+1, :, :])
        if device.type == "cuda":
            e1.record(); torch.cuda.synchronize(device)
            lats.append(e0.elapsed_time(e1))
        else:
            lats.append((time.perf_counter()-t0)*1000)

    m = sum(lats)/len(lats)
    s = (sum((l-m)**2 for l in lats)/len(lats))**0.5
    return {
        "T": T, "mode": "streaming_kvcache",
        "latency_mean_ms": round(m, 3), "latency_std_ms": round(s, 3),
        "latency_min_ms":  round(min(lats), 3), "latency_max_ms": round(max(lats), 3),
        "throughput_fps":  round(BATCH_SIZE*T*1000/m, 1),
        "peak_gpu_mb":     round(_bytes_to_mb(torch.cuda.max_memory_allocated(device)) if device.type=="cuda" else 0, 2),
        "ram_delta_mb":    round(max(get_ram_mb()-ram0, 0), 2),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────
def run_all() -> pd.DataFrame:
    devices = [torch.device("cpu")]
    if torch.cuda.is_available():
        devices.append(torch.device("cuda"))

    rows: List[Dict] = []

    for device in devices:
        dev_name = torch.cuda.get_device_name(device) if device.type=="cuda" else "CPU"
        print(f"\n{'='*72}")
        print(f"  Device : {dev_name}  ({device})")
        print(f"{'='*72}")

        model_kv = THCTNetKV(NUM_CLASSES).to(device)
        model_kv.eval()
        n_p = count_params(model_kv)
        print(f"  THCTNetKV params : {n_p:,}")

        print(f"\n  ── Batch mode (full sequence, same as original) ──")
        print(f"  {'T':>6}  {'mean ms':>10}  {'std ms':>7}  {'min ms':>8}  "
              f"{'max ms':>8}  {'FPS':>9}  {'peak GPU MB':>12}  {'ΔRAM MB':>9}")
        print(f"  {'-'*87}")
        for T in FRAME_LENGTHS:
            gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
            try:
                r = benchmark_batch(model_kv, T, device)
            except RuntimeError as e:
                print(f"  T={T} error: {e}"); continue
            print(f"  {T:>6}  {r['latency_mean_ms']:>10.3f}  {r['latency_std_ms']:>7.3f}  "
                  f"{r['latency_min_ms']:>8.3f}  {r['latency_max_ms']:>8.3f}  "
                  f"{r['throughput_fps']:>9.1f}  {r['peak_gpu_mb']:>12.2f}  {r['ram_delta_mb']:>9.2f}")
            rows.append({"device": device.type, "device_name": dev_name, **r})

        print(f"\n  ── Streaming / KV-cache mode (transformer only, token by token) ──")
        print(f"  {'T':>6}  {'mean ms':>10}  {'std ms':>7}  {'min ms':>8}  "
              f"{'max ms':>8}  {'FPS':>9}  {'peak GPU MB':>12}  {'ΔRAM MB':>9}")
        print(f"  {'-'*87}")
        for T in FRAME_LENGTHS:
            gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
            try:
                r = benchmark_streaming(model_kv, T, device)
            except RuntimeError as e:
                print(f"  T={T} error: {e}"); continue
            print(f"  {T:>6}  {r['latency_mean_ms']:>10.3f}  {r['latency_std_ms']:>7.3f}  "
                  f"{r['latency_min_ms']:>8.3f}  {r['latency_max_ms']:>8.3f}  "
                  f"{r['throughput_fps']:>9.1f}  {r['peak_gpu_mb']:>12.2f}  {r['ram_delta_mb']:>9.2f}")
            rows.append({"device": device.type, "device_name": dev_name, **r})

        del model_kv
        gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv("results/benchmark_kvcache.csv", index=False)
    print("\n  Saved → results/benchmark_kvcache.csv")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Plotting
# ─────────────────────────────────────────────────────────────────────────────
def plot_comparison(df_kv: pd.DataFrame):
    if not MPL:
        return

    orig_path = "results/benchmark_original.csv"
    if os.path.exists(orig_path):
        df_orig = pd.read_csv(orig_path)
        # Label the modes
        df_orig["mode"] = "original"
        combined = pd.concat([df_orig, df_kv], ignore_index=True)
    else:
        combined = df_kv.copy()

    devices = combined["device"].unique()

    fig, axes = plt.subplots(len(devices), 2, figsize=(14, 5 * len(devices)))
    if len(devices) == 1:
        axes = axes[None, :]   # ensure 2-D indexing

    fig.suptitle("THCT-Net: Original vs KV-Cache Benchmark", fontsize=14, fontweight="bold")

    mode_styles = {
        "original":          ("o-",  "#4C72B0", "Original (batch)"),
        "batch":             ("s--", "#55A868", "KV-Cache (batch)"),
        "streaming_kvcache": ("^-",  "#DD8452", "KV-Cache (streaming)"),
    }

    for row_i, dev in enumerate(devices):
        sub  = combined[combined["device"] == dev]
        modes = sub["mode"].unique()

        # Latency
        ax = axes[row_i, 0]
        for mode in modes:
            ms = sub[sub["mode"] == mode]
            fmt, col, lbl = mode_styles.get(mode, ("o-", "gray", mode))
            ax.plot(ms["T"], ms["latency_mean_ms"], fmt, color=col, label=lbl)
            if "latency_std_ms" in ms.columns:
                ax.fill_between(ms["T"],
                                ms["latency_mean_ms"] - ms["latency_std_ms"],
                                ms["latency_mean_ms"] + ms["latency_std_ms"],
                                alpha=0.12, color=col)
        ax.set_title(f"Latency — {dev.upper()}")
        ax.set_xlabel("Frame Length (T)"); ax.set_ylabel("Latency (ms)")
        ax.legend(); ax.grid(True, alpha=0.3)

        # Throughput
        ax = axes[row_i, 1]
        for mode in modes:
            ms  = sub[sub["mode"] == mode]
            fmt, col, lbl = mode_styles.get(mode, ("o-", "gray", mode))
            ax.plot(ms["T"], ms["throughput_fps"], fmt, color=col, label=lbl)
        ax.set_title(f"Throughput — {dev.upper()}")
        ax.set_xlabel("Frame Length (T)"); ax.set_ylabel("Frames / sec")
        ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("results/benchmark_comparison.png", dpi=150)
    plt.close()
    print("  Saved → results/benchmark_comparison.png")


# ─────────────────────────────────────────────────────────────────────────────
# Correctness check: KV-cache streaming ≈ batch (max |Δ| < 1e-4)
# ─────────────────────────────────────────────────────────────────────────────
def correctness_check(T: int = 10, device: torch.device = torch.device("cpu")):
    print("\n── Correctness check (streaming == batch for transformer stream) ──")
    model = THCTNetKV(NUM_CLASSES).to(device)
    model.eval()

    seq  = torch.randn(1, T, NUM_FEATURES, device=device)
    skel = _to_skeleton_tensor(seq)

    # Batch
    with torch.no_grad():
        batch_out = model.trans_stream(skel)   # (1, T, C)

    # Streaming
    model.trans_stream.reset_cache(1, device)
    stream_outs = []
    with torch.no_grad():
        for t in range(T):
            o = model.trans_stream.forward_step(skel[:, :, t:t+1, :, :])
            stream_outs.append(o)
    stream_out = torch.cat(stream_outs, dim=1)   # (1, T, C)

    diff = (batch_out - stream_out).abs().max().item()
    status = "PASSED ✓" if diff < 1e-3 else "FAILED ✗"
    print(f"  Max |Δ| between batch and streaming output : {diff:.2e}  →  {status}")


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    correctness_check(T=20, device=torch.device("cpu"))
    df_kv = run_all()
    plot_comparison(df_kv)

    # Also save merged comparison CSV
    orig_path = "results/benchmark_original.csv"
    if os.path.exists(orig_path):
        df_orig = pd.read_csv(orig_path)
        df_orig["mode"] = "original"
        merged = pd.concat([df_orig, df_kv], ignore_index=True)
        merged.to_csv("results/benchmark_comparison.csv", index=False)
        print("  Saved → results/benchmark_comparison.csv")

    print("\nAll done.\n")
    print("Recommended run order:")
    print("  1.  python benchmark_thctnet.py       # baseline")
    print("  2.  python benchmark_kvcache.py       # KV-cache + comparison")


── Correctness check (streaming == batch for transformer stream) ──
  Max |Δ| between batch and streaming output : 7.75e-07  →  PASSED ✓

  Device : CPU  (cpu)
  THCTNetKV params : 1,277,759

  ── Batch mode (full sequence, same as original) ──
       T     mean ms   std ms    min ms    max ms        FPS   peak GPU MB    ΔRAM MB
  ---------------------------------------------------------------------------------------
       5      26.468    3.537    21.307    34.428      188.9          0.00       1.06
      50      75.668   26.375    49.197   129.549      660.8          0.00       0.04
     100     125.441   40.747    76.261   212.869      797.2          0.00       0.00
     200     169.189   28.221   143.769   226.138     1182.1          0.00       3.99
     300     223.559   14.151   207.918   255.752     1341.9          0.00       0.00
     400     293.861   14.798   271.223   318.311     1361.2          0.00       0.08
     500     378.854   47.098   330.093   510.726     1319.8  

KeyboardInterrupt: 

In [1]:

"""
benchmark_kvcache.py
════════════════════
Benchmarks THCT-Net with a KV-cached transformer stream.

STREAMING LATENCY — what this script measures
──────────────────────────────────────────────
A live sensor sends one skeleton frame at a time.
By the time frame T arrives, frames 0 … T-1 have ALREADY been
processed in real time and their K/V tensors are sitting in the cache.

So "latency at frame T" = time to run forward_step(frame T) only.
Past work is NOT counted — it already happened.

Each timed run (streaming):
  1. prefill cache with frames 0 … T-2   ← NOT timed
  2. START timer
  3. forward_step(frame T-1)              ← the new frame arriving now
  4. STOP timer

With a correct KV-cache the per-step cost is nearly CONSTANT across T
because only the attention reads grow (O(T)), while projection + FFN
are always O(1) frame.

Batch mode (original API) is also benchmarked for comparison.

Outputs (in ./results/)
────────────────────────
  benchmark_kvcache.csv        KV-cache results
  benchmark_kvcache.png        KV-cache charts
  benchmark_comparison.csv     merged with original baseline (if present)
  benchmark_comparison.png     side-by-side comparison chart

Run order
─────────
  pip install torch pandas psutil matplotlib
  python benchmark_thctnet.py    # baseline first
  python benchmark_kvcache.py
"""

import gc
import math
import os
import time
from typing import Dict, List, Optional, Tuple

import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    MPL = True
except ImportError:
    MPL = False

os.makedirs("results", exist_ok=True)

# ─────────────────────────────────────────────────────────────────────
# Settings
# ─────────────────────────────────────────────────────────────────────
FRAME_LENGTHS = [5, 50, 100, 200, 300, 400, 500, 600, 750]
NUM_CLASSES   = 21
BATCH_SIZE    = 1
WARMUP_RUNS   = 5
TIMED_RUNS    = 20

# ─────────────────────────────────────────────────────────────────────
# Model constants
# ─────────────────────────────────────────────────────────────────────
T_FRAMES     = 750
NUM_FEATURES = 132
M_ENTITIES   = 2
V_PER_ENT    = 22
C_IN         = 3
NUM_JOINTS   = V_PER_ENT * M_ENTITIES   # 44


# ═════════════════════════════════════════════════════════════════════
# Shared building blocks  (causal LN helpers, pad helpers)
# ═════════════════════════════════════════════════════════════════════
class CausalLN2d(nn.Module):
    def __init__(self, c, eps=1e-5):
        super().__init__(); self.ln = nn.LayerNorm(c, eps=eps)
    def forward(self, x):                          # (B, C, T, V)
        return self.ln(x.permute(0,2,3,1)).permute(0,3,1,2)

class CausalLN3d(nn.Module):
    def __init__(self, c, eps=1e-5):
        super().__init__(); self.ln = nn.LayerNorm(c, eps=eps)
    def forward(self, x):                          # (B, C, T, H, W)
        return self.ln(x.permute(0,2,3,4,1)).permute(0,4,1,2,3)

class CausalLN1d(nn.Module):
    def __init__(self, c, eps=1e-5):
        super().__init__(); self.ln = nn.LayerNorm(c, eps=eps)
    def forward(self, x):                          # (B, C, T)
        return self.ln(x.permute(0,2,1)).permute(0,2,1)

def _causal_pad1d(x, ks, d=1):
    return F.pad(x, ((ks-1)*d, 0))

def _causal_pad2d_time(x, kt, dt=1):
    return F.pad(x, (0, 0, (kt-1)*dt, 0))

def _to_skeleton_tensor(x: Tensor) -> Tensor:
    """(B, T, 132) → (B, 3, T, 22, 2)"""
    B, T, D = x.shape
    return x.reshape(B, T, M_ENTITIES, V_PER_ENT, C_IN).permute(0,4,1,3,2).contiguous()


# ═════════════════════════════════════════════════════════════════════
# KV-Cache container
# ═════════════════════════════════════════════════════════════════════
class KVCache:
    """
    Stores the growing K and V tensors for one attention block.
    After t steps: shape is (B, H, t, Dh) for both k and v.
    """
    def __init__(self):
        self.k: Optional[Tensor] = None
        self.v: Optional[Tensor] = None

    def reset(self):
        self.k = None
        self.v = None

    def update(self, new_k: Tensor, new_v: Tensor) -> Tuple[Tensor, Tensor]:
        """
        Append new_k / new_v  (B, H, 1, Dh)  and return full (B, H, t, Dh).
        """
        if self.k is None:
            self.k, self.v = new_k, new_v
        else:
            self.k = torch.cat([self.k, new_k], dim=2)
            self.v = torch.cat([self.v, new_v], dim=2)
        return self.k, self.v


# ═════════════════════════════════════════════════════════════════════
# KV-cache ISATA attention block
# ═════════════════════════════════════════════════════════════════════
class _KVISATABlock(nn.Module):
    """
    Causal ISATA block with optional KV-cache for streaming.

    forward(x, cache=None)
    ──────────────────────
    cache=None  → standard batch forward; full upper-triangular causal mask.
    cache given → single-token step; K/V appended to cache.
                  No causal mask needed — query attends only to past+self
                  (everything already in cache).
    """
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.alpha  = nn.Parameter(torch.ones(1))
        self.A      = nn.Parameter(torch.zeros(num_heads, 1, 1))
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: Tensor, cache: Optional[KVCache] = None) -> Tensor:
        B, Tq, d = x.shape
        H, Dh    = self.num_heads, self.head_dim
        residual = x
        x_n = self.norm1(x)

        Q  = self.q_proj(x_n).reshape(B, Tq, H, Dh).transpose(1, 2)   # (B,H,Tq,Dh)
        Kn = self.k_proj(x_n).reshape(B, Tq, H, Dh).transpose(1, 2)   # (B,H,Tq,Dh)
        Vn = x_n             .reshape(B, Tq, H, Dh).transpose(1, 2)   # (B,H,Tq,Dh)

        if cache is not None:
            # ── Streaming step ────────────────────────────────────────
            # Append new K/V to cache; attend over full history.
            # Tq == 1, so the single query naturally sees only past+self.
            # No mask needed.
            K, V = cache.update(Kn, Vn)
        else:
            # ── Batch mode ────────────────────────────────────────────
            K, V = Kn, Vn

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Dh)
        scores = self.alpha * torch.tanh(scores) + self.A

        if cache is None:
            # Standard upper-triangular causal mask using torch.where
            # (avoids the 0 * -inf = NaN pitfall of float multiplication)
            Tk   = K.shape[2]
            q_idx = torch.arange(Tq, device=x.device).unsqueeze(1)  # (Tq,1)
            k_idx = torch.arange(Tk, device=x.device).unsqueeze(0)  # (1, Tk)
            mask  = k_idx > q_idx                                     # (Tq,Tk) bool
            scores = torch.where(mask, torch.full_like(scores, float("-inf")), scores)

        attn = F.softmax(scores, dim=-1)
        out  = torch.matmul(attn, V).transpose(1, 2).reshape(B, Tq, d) + residual
        return out + self.ffn(self.norm2(out))


# ═════════════════════════════════════════════════════════════════════
# KV-cache Transformer stream
# ═════════════════════════════════════════════════════════════════════
class TransformerStreamKV(nn.Module):
    """
    Drop-in replacement for TransformerStream with KV-cache streaming.

    Batch mode (same API as original):
        out = stream.forward(x)           # (B,C,T,V,M) → (B,T,num_classes)

    Streaming mode (one frame at a time):
        stream.reset_cache(B, device)
        for each new frame x_t:           # (B,C,1,V,M)
            logit_t = stream.forward_step(x_t)  # (B,1,num_classes)
    """
    def __init__(self, num_classes, d_model=128, num_heads=4,
                 num_layers=4, dropout=0.1):
        super().__init__()
        self.d_model    = d_model
        self.num_layers = num_layers

        self.frame_embed = nn.Sequential(
            nn.Conv3d(C_IN, d_model,
                      kernel_size=(1, V_PER_ENT, M_ENTITIES),
                      stride=(1, V_PER_ENT, M_ENTITIES), bias=False),
            CausalLN3d(d_model), nn.GELU(),
        )
        self.pos_enc = nn.Parameter(torch.zeros(1, T_FRAMES, d_model))
        nn.init.trunc_normal_(self.pos_enc, std=0.02)

        self.blocks = nn.ModuleList([
            _KVISATABlock(d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.temporal_agg = nn.Conv1d(d_model, d_model, kernel_size=5, bias=False)
        self.agg_norm     = CausalLN1d(d_model)
        self.head         = nn.Linear(d_model, num_classes)

        # Streaming state — initialised by reset_cache()
        self._caches:     List[KVCache] = [KVCache() for _ in range(num_layers)]
        self._agg_buf:    Optional[Tensor] = None   # (B, d, 4) causal Conv1d buffer
        self._step_count: int = 0

    # ── Batch forward (identical semantics to original) ───────────────
    def forward(self, x: Tensor) -> Tensor:
        """(B, C, T, V, M) → (B, T, num_classes)"""
        B, C, T, V, M = x.shape
        tokens = self.frame_embed(x).squeeze(-1).squeeze(-1).transpose(1, 2)
        tokens = tokens + self.pos_enc[:, :T, :]
        for blk in self.blocks:
            tokens = blk(tokens, cache=None)
        t = _causal_pad1d(tokens.transpose(1, 2), 5)
        t = F.gelu(self.agg_norm(self.temporal_agg(t)))
        return self.head(t.transpose(1, 2))

    # ── Streaming interface ───────────────────────────────────────────
    def reset_cache(self, B: int, device: torch.device):
        """Call once at the start of a new sequence before forward_step."""
        for c in self._caches:
            c.reset()
        self._agg_buf    = torch.zeros(B, self.d_model, 4, device=device)
        self._step_count = 0

    def forward_step(self, x_step: Tensor) -> Tensor:
        """
        Process exactly ONE new frame.
          x_step  : (B, C, 1, V, M)
          returns : (B, 1, num_classes)

        K/V caches for all attention blocks are updated in-place.
        Call reset_cache() before the first frame of each new sequence.
        """
        t_idx = self._step_count

        # Embed the single frame
        token = self.frame_embed(x_step).squeeze(-1).squeeze(-1).transpose(1, 2)
        token = token + self.pos_enc[:, t_idx:t_idx+1, :]    # (B, 1, d)

        # Attend over full cached history (cache updated inside each block)
        for blk, cache in zip(self.blocks, self._caches):
            token = blk(token, cache=cache)                   # (B, 1, d)

        # Causal Conv1d with a 4-frame sliding buffer
        t_ch = token.transpose(1, 2)                          # (B, d, 1)
        ctx  = torch.cat([self._agg_buf, t_ch], dim=2)        # (B, d, 5)
        # detach to prevent graph from growing across steps
        self._agg_buf = ctx[:, :, 1:].detach()                # (B, d, 4)
        agg  = self.temporal_agg(ctx)                         # (B, d, 1)
        agg  = F.gelu(self.agg_norm(agg))

        self._step_count += 1
        return self.head(agg.transpose(1, 2))                  # (B, 1, num_classes)


# ═════════════════════════════════════════════════════════════════════
# CNN stream  (unchanged from original — no streaming API needed here)
# ═════════════════════════════════════════════════════════════════════
class _CausalCNNBranch(nn.Module):
    def __init__(self, base_ch=64):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(C_IN, base_ch, (1,1), bias=False),
            CausalLN2d(base_ch), nn.ReLU(inplace=True))
        self.enc2_conv = nn.Conv2d(base_ch, base_ch, (3,1), padding=0, bias=False)
        self.enc2_norm = CausalLN2d(base_ch)
        self.enc3_conv = nn.Conv2d(NUM_JOINTS, base_ch, (3,3),
                                   padding=(0,1), dilation=(2,1), bias=False)
        self.enc3_norm = CausalLN2d(base_ch)
        self.enc4_conv = nn.Conv2d(base_ch, base_ch, (3,3),
                                   padding=(0,1), bias=False)
        self.enc4_norm = CausalLN2d(base_ch)

    def forward(self, x):
        x = self.enc1(x)
        x = F.relu(self.enc2_norm(self.enc2_conv(_causal_pad2d_time(x, 3, 1))))
        x = x.permute(0,3,2,1).contiguous()
        x = F.relu(self.enc3_norm(self.enc3_conv(_causal_pad2d_time(x, 3, 2))))
        return F.relu(self.enc4_norm(self.enc4_conv(_causal_pad2d_time(x, 3, 1))))


class _CausalResidualFusion(nn.Module):
    def __init__(self, in_ch, out_ch=128):
        super().__init__()
        self.conv_s   = nn.Conv2d(in_ch, out_ch, (1,7), padding=(0,3), bias=False)
        self.norm_s   = CausalLN2d(out_ch)
        self.conv_t   = nn.Conv2d(out_ch, out_ch, (7,1), padding=0, bias=False)
        self.norm_t   = CausalLN2d(out_ch)
        self.shortcut = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                                       CausalLN2d(out_ch))
                         if in_ch != out_ch else nn.Identity())
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        skip = self.shortcut(x)
        out  = F.relu(self.norm_s(self.conv_s(x)))
        return self.act(self.norm_t(self.conv_t(_causal_pad2d_time(out, 7))) + skip)


class CNNStream(nn.Module):
    def __init__(self, num_classes, base_ch=64):
        super().__init__()
        self.branch_S = _CausalCNNBranch(base_ch)
        self.branch_M = _CausalCNNBranch(base_ch)
        self.fusion   = _CausalResidualFusion(base_ch*2, 128)
        self.fc1  = nn.Linear(128, 256)
        self.drop = nn.Dropout(0.3)
        self.fc2  = nn.Linear(256, num_classes)

    @staticmethod
    def _backward_diff(x):
        return F.pad(x[:,:,1:,:] - x[:,:,:-1,:], (0,0,1,0))

    def forward(self, x):
        B, C, T, V, M = x.shape
        flat  = x.reshape(B, C, T, V*M)
        mot   = self._backward_diff(flat)
        fS    = self.branch_S(flat)
        fM    = self.branch_M(mot)
        if fS.shape != fM.shape:
            fM = F.interpolate(fM, size=fS.shape[2:])
        fused = self.fusion(torch.cat([fS, fM], dim=1))
        out   = fused.mean(dim=-1).transpose(1, 2)
        return self.fc2(self.drop(F.relu(self.fc1(out))))


# ═════════════════════════════════════════════════════════════════════
# Top-level KV-cache model
# ═════════════════════════════════════════════════════════════════════
class THCTNetKV(nn.Module):
    """
    THCT-Net with KV-cached transformer stream.

    Batch mode (same API as original THCTNet):
        logits = model(sequences, lengths)      # (B, T, num_classes)

    Streaming mode (transformer only):
        model.reset_cache(B, device)
        for each new frame t:
            frame_t  = sequences[:, t:t+1, :]           # (B, 1, D)
            skel_t   = _to_skeleton_tensor(frame_t)     # (B, C, 1, V, M)
            logit_t  = model.trans_stream.forward_step(skel_t)
    """
    def __init__(self, num_classes, d_model=128, num_heads=4,
                 num_layers=4, base_ch=64):
        super().__init__()
        self.cnn_stream   = CNNStream(num_classes, base_ch)
        self.trans_stream = TransformerStreamKV(num_classes, d_model,
                                                num_heads, num_layers)
        self._raw_w = nn.Parameter(torch.zeros(1))

    def forward(self, sequences: Tensor, lengths=None) -> Tensor:
        x  = _to_skeleton_tensor(sequences)
        lc = self.cnn_stream(x)
        lt = self.trans_stream(x)
        w  = torch.sigmoid(self._raw_w)
        return w * lc + (1.0 - w) * lt

    def reset_cache(self, B: int, device: torch.device):
        self.trans_stream.reset_cache(B, device)


# ═════════════════════════════════════════════════════════════════════
# Benchmark utilities
# ═════════════════════════════════════════════════════════════════════
def _mb(b):     return b / (1024 ** 2)
def _ram():     return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
def _params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)


def _time_one_call(fn, device):
    """Time a single no_grad call; returns elapsed ms."""
    if device.type == "cuda":
        e0 = torch.cuda.Event(enable_timing=True)
        e1 = torch.cuda.Event(enable_timing=True)
        e0.record()
        with torch.no_grad(): fn()
        e1.record()
        torch.cuda.synchronize(device)
        return e0.elapsed_time(e1)
    else:
        t0 = time.perf_counter()
        with torch.no_grad(): fn()
        return (time.perf_counter() - t0) * 1000


def _stats(lats: List[float], T: int, fps_frames: int) -> Dict:
    """Compute summary statistics from a list of latency samples."""
    m = sum(lats) / len(lats)
    s = (sum((l - m) ** 2 for l in lats) / len(lats)) ** 0.5
    return {
        "latency_mean_ms": round(m, 3),
        "latency_std_ms":  round(s, 3),
        "latency_min_ms":  round(min(lats), 3),
        "latency_max_ms":  round(max(lats), 3),
        # fps_frames = how many logical frames one call represents
        "throughput_fps":  round(fps_frames * 1000 / m, 1),
    }


def benchmark_batch(model: THCTNetKV, T: int, device: torch.device) -> Dict:
    """
    Full-sequence batch forward.
    Equivalent to the original model — all T frames processed at once.
    """
    model.eval()
    seq = torch.randn(BATCH_SIZE, T, NUM_FEATURES, device=device)

    # Warmup
    for _ in range(WARMUP_RUNS):
        with torch.no_grad(): model(seq)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    ram0 = _ram()

    # Timed runs
    lats = [_time_one_call(lambda: model(seq), device)
            for _ in range(TIMED_RUNS)]

    result = _stats(lats, T, fps_frames=BATCH_SIZE * T)
    result.update({
        "T":    T,
        "mode": "kvcache_batch",
        "peak_gpu_mb":  round(_mb(torch.cuda.max_memory_allocated(device))
                              if device.type == "cuda" else 0, 2),
        "ram_delta_mb": round(max(_ram() - ram0, 0), 2),
    })
    return result


def benchmark_streaming(model: THCTNetKV, T: int, device: torch.device) -> Dict:
    """
    TRUE single-step streaming latency at context length T.

    Measures ONLY the time to run forward_step(frame T-1),
    AFTER the cache has been pre-filled with frames 0 … T-2.

    The pre-fill is NOT timed — it represents work that already
    happened in real time as previous frames arrived from the sensor.

    Each timed run:
      ┌──────────────────────────────────────────────────────────┐
      │  prefill: forward_step(0) … forward_step(T-2)  ← NOT timed │
      │  ┌─ start timer ─────────────────────────────────────┐   │
      │  │  forward_step(T-1)   ← the new frame arriving now │   │
      │  └─ stop timer ──────────────────────────────────────┘   │
      └──────────────────────────────────────────────────────────┘

    If T=1 there is no pre-fill (first frame ever), which is valid.
    """
    model.eval()
    seq  = torch.randn(BATCH_SIZE, T, NUM_FEATURES, device=device)
    skel = _to_skeleton_tensor(seq)                # (B, C, T, V, M)

    def _prefill():
        """Fill cache with frames 0 … T-2.  Not timed."""
        model.trans_stream.reset_cache(BATCH_SIZE, device)
        with torch.no_grad():
            for t in range(T - 1):
                model.trans_stream.forward_step(skel[:, :, t:t+1, :, :])
        if device.type == "cuda":
            torch.cuda.synchronize(device)

    def _step():
        """Process only the final (new) frame.  This is what we time."""
        model.trans_stream.forward_step(skel[:, :, T-1:T, :, :])

    # Warmup
    for _ in range(WARMUP_RUNS):
        _prefill()
        with torch.no_grad(): _step()

    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    ram0 = _ram()

    # Timed runs: pre-fill (not timed) then time the single step
    lats = []
    for _ in range(TIMED_RUNS):
        _prefill()                                 # ← NOT in timer
        lats.append(_time_one_call(_step, device)) # ← only this is timed

    # throughput = 1 frame per step → fps = 1000 / latency_ms
    result = _stats(lats, T, fps_frames=BATCH_SIZE * 1)
    result.update({
        "T":    T,
        "mode": "kvcache_streaming",
        "peak_gpu_mb":  round(_mb(torch.cuda.max_memory_allocated(device))
                              if device.type == "cuda" else 0, 2),
        "ram_delta_mb": round(max(_ram() - ram0, 0), 2),
    })
    return result


# ═════════════════════════════════════════════════════════════════════
# Correctness check
# ═════════════════════════════════════════════════════════════════════
def correctness_check(T: int = 30, device: torch.device = torch.device("cpu")):
    """
    Verify that forward_step() token-by-token matches batch forward().
    Max |Δ| should be < 1e-3 (float32 rounding only).
    """
    print("\n── Correctness check ──────────────────────────────────────────")
    print(f"   batch vs streaming on T={T} frames, device={device}")
    model = THCTNetKV(NUM_CLASSES).to(device)
    model.eval()

    seq  = torch.randn(1, T, NUM_FEATURES, device=device)
    skel = _to_skeleton_tensor(seq)

    # Batch output
    with torch.no_grad():
        batch_out = model.trans_stream(skel)          # (1, T, C)

    # Streaming output (one frame at a time)
    model.trans_stream.reset_cache(1, device)
    stream_outs = []
    with torch.no_grad():
        for t in range(T):
            o = model.trans_stream.forward_step(skel[:, :, t:t+1, :, :])
            stream_outs.append(o)
    stream_out = torch.cat(stream_outs, dim=1)        # (1, T, C)

    diff = (batch_out - stream_out).abs().max().item()
    ok   = diff < 1e-3
    status = "PASSED ✓" if ok else "FAILED ✗"
    print(f"   Max |Δ| : {diff:.2e}  →  {status}")
    if not ok:
        raise RuntimeError("Streaming output does not match batch — check implementation.")
    print()


# ═════════════════════════════════════════════════════════════════════
# Main benchmark loop
# ═════════════════════════════════════════════════════════════════════
def run() -> pd.DataFrame:
    devices = [torch.device("cpu")]
    if torch.cuda.is_available():
        devices.append(torch.device("cuda"))

    rows: List[Dict] = []

    n_p = _params(THCTNetKV(NUM_CLASSES))
    print(f"\n{'='*74}")
    print(f"  THCT-Net + KV-Cache   {n_p:,} params")
    print(f"  Warmup={WARMUP_RUNS}  Timed={TIMED_RUNS}  Batch size={BATCH_SIZE}")
    print(f"{'='*74}")

    HDR = (f"  {'T':>6}  {'mean ms':>10}  {'std ms':>8}  "
           f"{'min ms':>8}  {'max ms':>8}  {'fps':>9}  "
           f"{'peak GPU MB':>12}  {'ΔRAM MB':>9}")
    SEP = f"  {'-'*91}"

    for device in devices:
        dev_name = (torch.cuda.get_device_name(device)
                    if device.type == "cuda" else "CPU")
        print(f"\n  Device : {dev_name}  ({device})")

        model = THCTNetKV(NUM_CLASSES).to(device)
        model.eval()

        # ── 1. Batch mode ─────────────────────────────────────────────
        print(f"\n  ── Batch mode (all T frames at once — same as original) ──")
        print(HDR); print(SEP)
        for T in FRAME_LENGTHS:
            gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
            r = benchmark_batch(model, T, device)
            print(f"  {T:>6}  {r['latency_mean_ms']:>10.3f}  {r['latency_std_ms']:>8.3f}  "
                  f"{r['latency_min_ms']:>8.3f}  {r['latency_max_ms']:>8.3f}  "
                  f"{r['throughput_fps']:>9.1f}  {r['peak_gpu_mb']:>12.2f}  "
                  f"{r['ram_delta_mb']:>9.2f}")
            rows.append({"device": device.type, "device_name": dev_name, **r})

        # ── 2. Streaming mode ─────────────────────────────────────────
        print(f"\n  ── KV-cache STREAMING mode ──")
        print(f"  T = context length already cached.  "
              f"Latency = time to process the ONE new frame arriving now.")
        print(f"  A correct KV-cache should give ~constant latency across all T.")
        print(HDR); print(SEP)
        for T in FRAME_LENGTHS:
            gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
            r = benchmark_streaming(model, T, device)
            print(f"  {T:>6}  {r['latency_mean_ms']:>10.3f}  {r['latency_std_ms']:>8.3f}  "
                  f"{r['latency_min_ms']:>8.3f}  {r['latency_max_ms']:>8.3f}  "
                  f"{r['throughput_fps']:>9.1f}  {r['peak_gpu_mb']:>12.2f}  "
                  f"{r['ram_delta_mb']:>9.2f}")
            rows.append({"device": device.type, "device_name": dev_name, **r})

        del model; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv("results/benchmark_kvcache.csv", index=False)
    print(f"\n  Saved → results/benchmark_kvcache.csv")
    return df


# ═════════════════════════════════════════════════════════════════════
# Plotting
# ═════════════════════════════════════════════════════════════════════
def _plot_kvcache(df: pd.DataFrame):
    if not MPL: return
    devices = df["device"].unique()
    fig, axes = plt.subplots(len(devices), 2, figsize=(14, 5 * len(devices)))
    if len(devices) == 1: axes = axes[None, :]
    fig.suptitle("THCT-Net KV-Cache Benchmark\n"
                 "(streaming latency = single forward_step after pre-fill)",
                 fontsize=13, fontweight="bold")

    styles = {
        "kvcache_batch":     ("s--", "#55A868", 0.7, "KV-Cache batch (all T at once)"),
        "kvcache_streaming": ("o-",  "#DD8452", 1.0,
                              "KV-Cache streaming\n(latency of new frame at context T)"),
    }

    for i, dev in enumerate(devices):
        sub = df[df["device"] == dev]
        for mode, (fmt, col, alpha, lbl) in styles.items():
            ms = sub[sub["mode"] == mode]
            if ms.empty: continue
            axes[i,0].plot(ms["T"], ms["latency_mean_ms"],
                           fmt, color=col, alpha=alpha, label=lbl)
            axes[i,0].fill_between(
                ms["T"],
                ms["latency_mean_ms"] - ms["latency_std_ms"],
                ms["latency_mean_ms"] + ms["latency_std_ms"],
                alpha=0.1, color=col)
            axes[i,1].plot(ms["T"], ms["throughput_fps"],
                           fmt, color=col, alpha=alpha, label=lbl)

        axes[i,0].set_title(f"Latency — {dev.upper()}")
        axes[i,0].set_xlabel("T (context length at measurement)")
        axes[i,0].set_ylabel("ms"); axes[i,0].legend(); axes[i,0].grid(alpha=0.3)
        axes[i,1].set_title(f"Throughput — {dev.upper()}")
        axes[i,1].set_xlabel("T")
        axes[i,1].set_ylabel("frames / sec")
        axes[i,1].legend(); axes[i,1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("results/benchmark_kvcache.png", dpi=150)
    plt.close()
    print("  Saved → results/benchmark_kvcache.png")


def _plot_comparison(df_kv: pd.DataFrame):
    """Overlay original baseline (if available) vs KV-cache streaming."""
    if not MPL: return
    orig_path = "results/benchmark_original.csv"
    if not os.path.exists(orig_path):
        print("  (comparison plot skipped — run benchmark_thctnet.py first)")
        return

    df_orig = pd.read_csv(orig_path)
    df_orig["mode"] = "original_batch"
    combined = pd.concat([df_orig, df_kv], ignore_index=True)
    combined.to_csv("results/benchmark_comparison.csv", index=False)
    print("  Saved → results/benchmark_comparison.csv")

    devices = combined["device"].unique()
    styles  = {
        "original_batch":    ("o-",  "#4C72B0", "Original batch"),
        "kvcache_batch":     ("s--", "#55A868", "KV-Cache batch"),
        "kvcache_streaming": ("^-",  "#DD8452",
                              "KV-Cache streaming\n(latency at frame T, history cached)"),
    }

    fig, axes = plt.subplots(len(devices), 2, figsize=(14, 5 * len(devices)))
    if len(devices) == 1: axes = axes[None, :]
    fig.suptitle(
        "THCT-Net: Original vs KV-Cache\n"
        "Streaming = single-step latency at frame T (past frames already cached)",
        fontsize=12, fontweight="bold")

    for i, dev in enumerate(devices):
        sub = combined[combined["device"] == dev]
        for mode, (fmt, col, lbl) in styles.items():
            ms = sub[sub["mode"] == mode]
            if ms.empty: continue
            axes[i,0].plot(ms["T"], ms["latency_mean_ms"],
                           fmt, color=col, label=lbl)
            if "latency_std_ms" in ms.columns:
                axes[i,0].fill_between(
                    ms["T"],
                    ms["latency_mean_ms"] - ms["latency_std_ms"],
                    ms["latency_mean_ms"] + ms["latency_std_ms"],
                    alpha=0.1, color=col)
            axes[i,1].plot(ms["T"], ms["throughput_fps"],
                           fmt, color=col, label=lbl)

        axes[i,0].set_title(f"Latency — {dev.upper()}")
        axes[i,0].set_xlabel("T"); axes[i,0].set_ylabel("ms")
        axes[i,0].legend(); axes[i,0].grid(alpha=0.3)
        axes[i,1].set_title(f"Throughput — {dev.upper()}")
        axes[i,1].set_xlabel("T"); axes[i,1].set_ylabel("frames / sec")
        axes[i,1].legend(); axes[i,1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("results/benchmark_comparison.png", dpi=150)
    plt.close()
    print("  Saved → results/benchmark_comparison.png")


# ═════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    correctness_check(T=30)
    df_kv = run()
    _plot_kvcache(df_kv)
    _plot_comparison(df_kv)
    print("\nDone.\n")



── Correctness check ──────────────────────────────────────────
   batch vs streaming on T=30 frames, device=cpu
   Max |Δ| : 8.05e-07  →  PASSED ✓


  THCT-Net + KV-Cache   1,277,759 params
  Warmup=5  Timed=20  Batch size=1

  Device : CPU  (cpu)

  ── Batch mode (all T frames at once — same as original) ──
       T     mean ms    std ms    min ms    max ms        fps   peak GPU MB    ΔRAM MB
  -------------------------------------------------------------------------------------------
       5      35.962    10.948    22.044    63.876      139.0          0.00       0.00
      50      94.599    40.043    57.758   226.038      528.5          0.00       0.00
     100     104.004    14.427    87.227   153.633      961.5          0.00       0.00
     200     202.875    42.713   168.837   355.909      985.8          0.00       0.00
     300     280.935    24.613   227.953   323.888     1067.9          0.00       0.03
     400     388.607    50.834   322.924   505.634     1029.3          0